# Entanglement lab

Entanglement is the ledger for known pairs. It records where pair endpoints live, what state the pair is in, and how later code can find usable pairs.


In [ ]:
from simyuj.components import PortKind
from simyuj.entanglement import EntangledPairRecord, EntangledPairRegistry, PairState
from simyuj.entanglement.queries import (
    available_pairs_for_route_hops,
    available_pairs_touching_node,
    ordered_node_pair,
    pairs_by_node_pair,
    pairs_using_any_memory,
    route_hops_have_available_pairs,
)
from simyuj.network import Network, Node
from simyuj.resources import MemoryRef


## 1. Start with memory endpoints

Start with memory endpoints. A pair record points at resource-layer memory refs.


In [ ]:
def ref(node_id, position=0, device_id="qmem"):
    return MemoryRef(node_id, device_id, position)


alice_0 = ref("alice", 0)
bob_0 = ref("bob", 0)
relay_0 = ref("relay", 0)

print("Alice endpoint:", alice_0.key)
print("Bob endpoint:", bob_0.key)
print("Relay endpoint:", relay_0.key)


In [ ]:
pair = EntangledPairRecord(
    pair_id="pair:alice-bob:0",
    left=alice_0,
    right=bob_0,
    fidelity=0.94,
    created_at=100,
    expires_at=180,
    generation_link_id="q_alice_bob",
    metadata=(("source", "heralded-link"),),
)

print("Pair id:", pair.pair_id)
print("Endpoints:", pair.memory_ref_keys)
print("Nodes:", pair.node_ids)
print("State:", pair.state.value)
print("Fidelity:", pair.fidelity)
print("Generation link:", pair.generation_link_id)


## 2. Treat pair sides as storage labels

Left and right are storage labels. Node and memory matching treat the pair as undirected.


In [ ]:
print("Uses Alice memory:", pair.uses_memory(alice_0))
print("Other endpoint from Alice:", pair.other_memory(alice_0).key)
print("Has Bob node:", pair.has_node("bob"))
print("Connects Alice/Bob:", pair.connects_nodes("alice", "bob"))
print("Connects Bob/Alice:", pair.connects_nodes("bob", "alice"))
print("Connects exact refs reversed:", pair.connects_memory_refs(bob_0, alice_0))


## 3. Move records through states

State helpers make new records. The original record stays unchanged.


In [ ]:
reserved_copy = pair.reserved()
consumed_copy = pair.consumed()

print("Original state:", pair.state.value)
print("Reserved copy:", reserved_copy.state.value)
print("Consumed copy:", consumed_copy.state.value)
print("Original is active:", pair.is_active)
print("Consumed copy is terminal:", consumed_copy.is_terminal)


## 4. Put records in a registry

The registry stores records and owns lifecycle transitions.


In [ ]:
registry = EntangledPairRegistry()

registered = registry.register(pair)

print("Registered pair:", registered.pair_id)
print("Registry ids:", tuple(registry.pairs))
print("Available ids:", [p.pair_id for p in registry.available_pairs()])


In [ ]:
reserved = registry.reserve("pair:alice-bob:0")

print("After reserve:", reserved.state.value)
print("Available ids:", [p.pair_id for p in registry.available_pairs()])
print("Reserved ids:", [p.pair_id for p in registry.reserved_pairs()])


In [ ]:
released = registry.release("pair:alice-bob:0")

print("After release:", released.state.value)
print("Available ids:", [p.pair_id for p in registry.available_pairs()])


## 5. Block conflicting active pairs

An active memory position can belong to only one active pair.


In [ ]:
conflicting = EntangledPairRecord(
    pair_id="pair:conflict",
    left=alice_0,
    right=ref("carol", 0),
    fidelity=0.91,
)

try:
    registry.register(conflicting)
except ValueError as error:
    print("Conflict caught:", error)


## 6. Release memory after terminal states

Terminal records stay in history and no longer block memory reuse.


In [ ]:
consumed = registry.consume("pair:alice-bob:0")

replacement = registry.register(
    EntangledPairRecord(
        pair_id="pair:alice-carol:0",
        left=alice_0,
        right=ref("carol", 0),
        fidelity=0.90,
        created_at=190,
        expires_at=260,
        generation_link_id="q_alice_carol",
    )
)

print("Consumed old pair:", consumed.pair_id, consumed.state.value)
print("Replacement pair:", replacement.pair_id, replacement.state.value)
print("Active pair using Alice memory:", registry.pair_using_memory(alice_0).pair_id)
print("All pairs using Alice memory:", [p.pair_id for p in registry.pairs_using_memory(alice_0)])


## 7. Build a route-style inventory

Build a fuller registry for route-style lookup.


In [ ]:
lab = EntangledPairRegistry()

pairs = [
    EntangledPairRecord(
        "pair:ar:good",
        ref("alice", 0),
        ref("relay", 0),
        fidelity=0.93,
        created_at=10,
        expires_at=80,
        generation_link_id="q_alice_relay",
    ),
    EntangledPairRecord(
        "pair:ar:weak",
        ref("alice", 1),
        ref("relay", 1),
        fidelity=0.82,
        created_at=12,
        expires_at=70,
        generation_link_id="q_alice_relay",
    ),
    EntangledPairRecord(
        "pair:rb:good",
        ref("relay", 2),
        ref("bob", 0),
        fidelity=0.91,
        created_at=11,
        expires_at=75,
        generation_link_id="q_relay_bob",
    ),
]


In [ ]:
pairs.extend(
    [
        EntangledPairRecord(
            "pair:ab:direct",
            ref("alice", 2),
            ref("bob", 1),
            fidelity=0.86,
            created_at=15,
            expires_at=60,
            generation_link_id="q_alice_bob",
        ),
        EntangledPairRecord(
            "pair:old:ab",
            ref("alice", 3),
            ref("bob", 2),
            state=PairState.CONSUMED,
            fidelity=0.95,
            generation_link_id="q_alice_bob",
        ),
    ]
)

for item in pairs:
    lab.register(item)

print("All pair ids:", [p.pair_id for p in lab.all_pairs()])
print("Available pair ids:", [p.pair_id for p in lab.available_pairs()])


In [ ]:
print("Available pairs touching Alice, min fidelity 0.90:")
for item in available_pairs_touching_node(lab, "alice", min_fidelity=0.90):
    print(" ", item.pair_id, item.node_ids, item.fidelity)


In [ ]:
print("Available Alice/Bob pairs:")
for item in lab.available_between("alice", "bob"):
    print(" ", item.pair_id, "fidelity", item.fidelity, "link", item.generation_link_id)

print("Available Alice/Bob pairs with fidelity >= 0.90:")
print([p.pair_id for p in lab.available_between("alice", "bob", min_fidelity=0.90)])


In [ ]:
print("Alice/Bob pairs generated by q_alice_bob:")
print([p.pair_id for p in lab.available_between("alice", "bob", link_id="q_alice_bob")])


## 8. Query each route hop

Route-hop queries ask: does each hop already have usable entanglement?


In [ ]:
network = Network("entanglement_route")
for node_id in ("alice", "relay", "bob"):
    network.add_node(Node(node_id))

network.add_quantum_link("q_alice_relay", "alice", "relay")
network.add_quantum_link("q_relay_bob", "relay", "bob")
network.add_quantum_link("q_alice_bob", "alice", "bob")

relay_route = network.fewest_hops_path("alice", "bob", port_kind=PortKind.QUANTUM)

print("Fewest-hop route links:", relay_route.link_ids)


In [ ]:
# Pick the two-hop route explicitly so each hop maps to one link id.
two_hop_route = network.paths_with_max_hops(
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    max_hops=2,
)[1]

print("Two-hop route nodes:", two_hop_route.node_ids)
print("Two-hop route links:", two_hop_route.link_ids)


In [ ]:
hop_candidates = available_pairs_for_route_hops(
    lab,
    two_hop_route,
    min_fidelity=0.90,
)

for hop in hop_candidates:
    print(
        f"{hop.source_node_id} -> {hop.target_node_id}:",
        [pair.pair_id for pair in hop.pairs],
    )

print("Every hop has a candidate:", route_hops_have_available_pairs(lab, two_hop_route, min_fidelity=0.90))


## 9. Reserve deliberately after reading

Route queries are read-only. Reserve or consume deliberately afterward.


In [ ]:
chosen_ids = [hop.pairs[0].pair_id for hop in hop_candidates]

print("Chosen ids:", chosen_ids)
print("States before:", [(pair_id, lab.get(pair_id).state.value) for pair_id in chosen_ids])

for pair_id in chosen_ids:
    lab.reserve(pair_id)

print("States after reserve:", [(pair_id, lab.get(pair_id).state.value) for pair_id in chosen_ids])


In [ ]:
print("Two-hop route still has available candidates after reserve:")
refreshed = available_pairs_for_route_hops(lab, two_hop_route, min_fidelity=0.90)
for hop in refreshed:
    print(f"{hop.source_node_id} -> {hop.target_node_id}:", [p.pair_id for p in hop.pairs])


## 10. Group by node pair

Grouping by node pair gives a quick network-level inventory.


In [ ]:
grouped = pairs_by_node_pair(lab, min_fidelity=0.85)

print("Node-pair groups, fidelity >= 0.85:")
for key, items in grouped.items():
    print(" ", key, "->", [(item.pair_id, item.state.value, item.fidelity) for item in items])

print("Ordered key example:", ordered_node_pair("bob", "alice"))


## 11. Check memory history

History lookup is useful when a memory position is reused.


In [ ]:
history = pairs_using_any_memory(lab, (ref("alice", 0), ref("bob", 2)))

print("Pairs using Alice position 0 or Bob position 2:")
for item in history:
    print(" ", item.pair_id, item.memory_ref_keys, item.state.value)


## 12. Sweep expired active pairs

Expiry sweeps active records. Terminal records are left alone.


In [ ]:
expiring = EntangledPairRegistry()
expiring.register(EntangledPairRecord("pair:soon", ref("alice", 0), ref("bob", 0), expires_at=20))
expiring.register(EntangledPairRecord("pair:later", ref("alice", 1), ref("bob", 1), expires_at=40))
expiring.register(EntangledPairRecord("pair:old", ref("alice", 2), ref("bob", 2), state=PairState.CONSUMED, expires_at=1))

expired = expiring.expire_before(20)

print("Expired now:", [(item.pair_id, item.state.value) for item in expired])
print("Registry states:", [(item.pair_id, item.state.value) for item in expiring.all_pairs()])


## Keep this model in your head

The habit to keep: entanglement records describe known pairs. Query helpers help choose candidates, but pair selection, reservation, and consumption stay explicit.
